# ABC Selective Attention - Feature-Sensitive Text Counting

Kaggle Bench runner for the counting task on the feature-sensitive text slice.


In [1]:
from dataclasses import dataclass

import kaggle_benchmarks as kbench
import pandas as pd

import abc_selective_attention_utils as utils

In [2]:
FAMILY = "selective attention"
ATTENTIONAL_BASIS = "feature sensitive"
MODALITY = "text"
TASK_TYPE = "counting"
TASK_NAME = f"{FAMILY} - {ATTENTIONAL_BASIS} {MODALITY} {TASK_TYPE}"

In [3]:
@dataclass
class CountAnswer:
    count: int

In [4]:
DATASET_ROOT = utils.DEFAULT_DATASET_ROOT
CSV_PATH = DATASET_ROOT / 'selective_attention/feature_sensitive/text/feature_sensitive_text_full.csv'

df = utils.load_csv(CSV_PATH)
print(CSV_PATH)
print('rows:', len(df))
print('columns:', sorted(df.columns.tolist()))


/kaggle/input/datasets/patrycjawegrzynowicz/abc-factorized-attention-benchmark/selective_attention/feature_sensitive/text/feature_sensitive_text_full.csv
rows: 780
columns: ['active_fields', 'attentional_basis', 'count_instruction', 'count_prompt', 'dimension', 'family', 'filter_instruction', 'filter_prompt', 'gold_count', 'gold_lines', 'irrelevant_noise_fields', 'modality', 'num_records', 'position_mode', 'same_color_wrong_shape_count', 'same_core_wrong_marker_count', 'same_core_wrong_pattern_count', 'same_core_wrong_size_count', 'same_shape_wrong_color_count', 'seed', 'target_count', 'target_definition', 'target_feature_count', 'target_fields', 'text_input', 'unrelated_count', 'variant']


In [5]:
base_cols = [
    'seed',
    'family',
    'attentional_basis',
    'modality',
    'dimension',
    'variant',
    'num_records',
    'target_feature_count',
    'position_mode',
]

optional_cols = [
    col
    for col in [
        'target_count',
        'active_fields',
        'target_fields',
        'irrelevant_noise_fields',
        'same_color_wrong_shape_count',
        'same_shape_wrong_color_count',
        'same_core_wrong_marker_count',
        'same_core_wrong_size_count',
        'same_core_wrong_pattern_count',
        'unrelated_count',
    ]
    if col in df.columns
]

failure_cols = [
    'seed',
    'dimension',
    'variant',
    'gold_count',
    'predicted_count',
    'failure_type',
]

count_eval_df = df[base_cols + optional_cols + ['count_prompt', 'gold_count']].copy()
groupings = utils.build_text_groupings(df)

print('count_eval_df columns:', count_eval_df.columns.tolist())
print('groupings:', groupings)


count_eval_df columns: ['seed', 'family', 'attentional_basis', 'modality', 'dimension', 'variant', 'num_records', 'target_feature_count', 'position_mode', 'target_count', 'active_fields', 'target_fields', 'irrelevant_noise_fields', 'same_color_wrong_shape_count', 'same_shape_wrong_color_count', 'same_core_wrong_marker_count', 'same_core_wrong_size_count', 'same_core_wrong_pattern_count', 'unrelated_count', 'count_prompt', 'gold_count']
groupings: [['family'], ['family', 'attentional_basis'], ['family', 'attentional_basis', 'modality'], ['dimension'], ['dimension', 'variant'], ['target_feature_count'], ['num_records'], ['position_mode'], ['target_count'], ['dimension', 'target_count']]


In [6]:
@kbench.task(store_task=False)
def single_feature_sensitive_text_count(
    llm,
    seed,
    family,
    attentional_basis,
    modality,
    dimension,
    variant,
    num_records,
    target_feature_count,
    position_mode,
    count_prompt,
    gold_count,
    target_count=None,
    active_fields=None,
    target_fields=None,
    irrelevant_noise_fields=None,
    same_color_wrong_shape_count=None,
    same_shape_wrong_color_count=None,
    same_core_wrong_marker_count=None,
    same_core_wrong_size_count=None,
    same_core_wrong_pattern_count=None,
    unrelated_count=None,
) -> dict:
    gold = int(gold_count)

    pred = None
    error = None
    error_type = None

    try:
        response = llm.prompt(count_prompt, schema=CountAnswer)
        pred = int(response.count)
    except Exception as exc:
        error_type, error = utils.classify_prompt_error(exc)

    is_correct = pred == gold
    has_error = error_type is not None
    is_failure = not is_correct
    failure_type = 'ok' if is_correct else (error_type if has_error else 'wrong_answer')

    kbench.assertions.assert_true(
        is_correct,
        expectation=f'Expected {gold}, got {pred}' + (f' | error_type={error_type} | error={error}' if error else ''),
    )

    result = {
        'task': 'feature_sensitive_text_count_v1',
        'model': llm.name,
        'seed': int(seed),
        'family': family,
        'attentional_basis': attentional_basis,
        'modality': modality,
        'dimension': dimension,
        'variant': variant,
        'task_type': 'counting',
        'target_feature_count': int(target_feature_count),
        'num_records': int(num_records),
        'target_count': int(target_count) if target_count not in (None, '') else None,
        'position_mode': position_mode,
        'gold_count': gold,
        'predicted_count': pred,
        'is_correct': is_correct,
        'is_failure': is_failure,
        'has_error': has_error,
        'failure_type': failure_type,
    }

    if error_type is not None:
        result['error_type'] = error_type
    if error is not None:
        result['error'] = error
    return result


In [7]:
@kbench.task(name="selective attention - feature sensitive text counting")
def feature_sensitive_text_count_dataset(llm, df) -> tuple[float, float]:
    with kbench.client.enable_cache():
        runs = single_feature_sensitive_text_count.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            retry_delay=15,
            llm=[llm],
            evaluation_data=df,
            n_jobs=8,
            timeout=120,
            remove_run_files=False,
        )

    result = utils.summarize_and_log_runs(TASK_NAME, llm.name, runs, groupings, failure_cols)
    return result

In [8]:
run = feature_sensitive_text_count_dataset.run(kbench.llm, count_eval_df)
print(run)

=== selective attention - feature sensitive text counting : info ===
task: selective attention - feature sensitive text counting
model: qwen/qwen3-next-80b-a3b-instruct
timestamp: 20260414_020040
run id: qwen_qwen3_next_80b_a3b_instruct_20260414_020040
errors: 0 (rate 0.0)
overall: 556/780 = 71.28% (std 45.27%)
=== selective attention - feature sensitive text counting: by family ===
             family  passed  total  accuracy
selective_attention     556    780  0.712821
=== selective attention - feature sensitive text counting: by family, attentional_basis ===
             family attentional_basis  passed  total  accuracy
selective_attention feature_sensitive     556    780  0.712821
=== selective attention - feature sensitive text counting: by family, attentional_basis, modality ===
             family attentional_basis modality  passed  total  accuracy
selective_attention feature_sensitive     text     556    780  0.712821
=== selective attention - feature sensitive text counting: b

Run(task=Task(func=<function feature_sensitive_text_count_dataset at 0x7ab93f3b99e0>, name='selective attention - feature sensitive text counting', description='', result_type=<class 'kaggle_benchmarks.results.MetricWithCI'>, version=1, store_task=True, store_run=True), result=(0.7128205128205128, 0.45273635997723477), chat=Chat(history=[], name='selective attention - feature sensitive text counting', _id_suffix='0eb003f4', sender=Actor(name='System', avatar='⚙️'), _status=<Status.SUCCESS: 'success'>), status=<Status.SUCCESS: 'success'>, params={'llm': OpenAI(name='qwen/qwen3-next-80b-a3b-instruct'), 'df':      seed               family  attentional_basis modality dimension  \
0    1000  selective_attention  feature_sensitive     text  baseline   
1    1001  selective_attention  feature_sensitive     text  baseline   
2    1002  selective_attention  feature_sensitive     text  baseline   
3    1003  selective_attention  feature_sensitive     text  baseline   
4    1004  selective_atten

In [9]:
%choose feature_sensitive_text_count_dataset


Kept: selective_attention_-_feature_sensitive_text_counting.task.json
Kept: selective_attention_-_feature_sensitive_text_counting-run_id_Run_1_qwen_qwen3-next-80b-a3b-instruct.run.json
